In [1]:
import os

In [2]:
%pwd

'c:\\Users\\chauh\\OneDrive\\Desktop\\Text-Summarization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\chauh\\OneDrive\\Desktop\\Text-Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE__PATH):
        
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)


        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,       # Must match entity
            tokenizer_path=config.tokenizer_path, # Must match entity
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config
        
    

In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from datasets import load_dataset, load_from_disk
import torch
import tqdm

[2026-05-01 19:12:06,646: WARNING:  module_wrapper: From c:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
 ]
[2026-05-01 19:12:07,601: INFO:  config: TensorFlow version 2.21.0 available. ]


In [8]:
class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config=config

    def generate_batch_sized_chunks(self,List_of_elements,batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0,len(List_of_elements),batch_size):
            yield List_of_elements[i: i+batch_size]

    def calculate_metric_on_test_ds(self,dataset,metrics,model,tokenizer,
                                    batch_size=16,device="cuda" if torch.cuda.is_available() else "cpu",
                                    column_text="article",
                                    column_summary="highlights"):
        article_batches=list(self.generate_batch_sized_chunks(dataset[column_text],batch_size))
        target_batches=list(self.generate_batch_sized_chunks(dataset[column_summary],batch_size))

        for article_batch,target_batch in tqdm(
            zip(article_batches,target_batches), total=len(article_batches)):

            inputs=tokenizer(article_batch,max_length=1024,truncation=True,padding="max_length",return_tensors="pt")

            summaries=model.generate(input_ids=inputs["input_ids"].to(device),
                                     attention_mask=inputs["attention_mask"].to(device),
                                     Length_penalty=0.8,num_beams=8,max_length=128)
            '''parameter for length penalty ensures that the model does not generate sequences that are too long.'''

            # Finally, we decode the generate texts
            # replace the token,add the decoded texts with the references to the metrics.
            decoded_summaries=[tokenizer.decode(s, skip_special_tokens=True,clean_up_tokenization_spaces=True)
                               for s in summaries]
            decoded_summaries=[d.replace(""," ") for d in decoded_summaries]

            metrics.add_batch(predictions=decoded_summaries,references=target_batch)

        # finally compute and return the rouge score
        score=metrics.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)


        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
  
        rouge_metric = load_metric('rouge')

        score = self.calculate_metric_on_test_ds(
        dataset_samsum_pt['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
            )

        rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )

        df = pd.DataFrame(rouge_dict, index = ['pegasus'] )
        df.to_csv(self.config.metric_file_name, index=False)

In [9]:
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk # Removed load_metric from here
import evaluate 
import torch
import pandas as pd


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the dataset into smaller batches to save RAM."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metrics, model, tokenizer, 
                                   batch_size=1, # Set to 1 for CPU stability
                                   device="cpu", 
                                   column_text="dialogue", 
                                   column_summary="summary"):
        
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
            
            # T5 requirement: Prepend "summarize: " to the input text
            inputs_text = ["summarize: " + doc for doc in article_batch]
            
            # Lower max_length to 128 to prevent Kernel Crash
            inputs = tokenizer(inputs_text, max_length=128, truncation=True, 
                               padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                       attention_mask=inputs["attention_mask"].to(device),
                                       length_penalty=0.8, 
                                       num_beams=4, # Reduced from 8 to save CPU cycles
                                       max_length=64) # Summary doesn't need to be 128

            # Decode the generated texts
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                                                clean_up_tokenization_spaces=True) 
                                 for s in summaries]

            metrics.add_batch(predictions=decoded_summaries, references=target_batch)

        # Compute and return the rouge score
        score = metrics.compute()
        return score
    # In src/textsummarizer/components/model_evaluation.py

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Adding local_files_only=True is key here
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path, 
            local_files_only=True
        )
        model_t5 = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path, 
            local_files_only=True
        ).to(device)
        
        # ... rest of the code

   
       
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # UPDATED: Use evaluate.load instead of load_metric
        rouge_metric = evaluate.load('rouge')

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:10], 
            rouge_metric, 
            model_t5, 
            tokenizer, 
            batch_size = 1, 
            column_text = 'dialogue', 
            column_summary= 'summary'
        )

        # UPDATED: 'evaluate' library returns scores as a simple dictionary of floats
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_dict = {rn: score[rn] for rn in rouge_names}

        df = pd.DataFrame(rouge_dict, index = ['t5-small'])
        df.to_csv(self.config.metric_file_name, index=False)

In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-05-01 19:12:17,934: INFO:  common: yaml file: config\config.yaml loaded successfully ]


[2026-05-01 19:12:17,940: INFO:  common: yaml file: params.yaml loaded successfully ]
[2026-05-01 19:12:17,945: INFO:  common: created directory at: artifacts ]
[2026-05-01 19:12:17,947: INFO:  common: created directory at: artifacts/model_evaluation ]


100%|██████████| 10/10 [00:35<00:00,  3.57s/it]

[2026-05-01 19:13:00,629: INFO:  rouge_scorer: Using default tokenizer. ]
